# RUNG — test every algorithm

Run this notebook on Kaggle with **Internet** and **GPU** turned on (Settings → Accelerator → GPU, then turn Internet on). Session output, logs, and charts are written under `/kaggle/working`.

The notebook loads [TAR2003/RUNG_ML_Project](https://github.com/TAR2003/RUNG_ML_Project). It then runs the unit checks, clean training, the structure PGD attack, and the cosine adaptive attack for every algorithm on every dataset, and draws the charts.

**Push your local checkout before you rely on the GitHub clone.** Presets S01–S34 live in this working tree until they are on `master`. Two ways to get that code onto Kaggle:

1. Push `master` to GitHub, then run this notebook. It clones that remote.
2. Zip the project and upload it as a Kaggle dataset, then attach the dataset to this notebook. If the attachment contains `model/rung_strategy.py`, that copy is used.

A GPU session lasts about 12 hours, which is shorter than the full grid. Each finished dataset/algorithm pair is skipped on the next run when its logs are still present. To continue in a new session, download `rung_results.zip` from the notebook output, upload it as a dataset, and attach it. The setup cell unpacks `log/`, `figures/`, and `exp/models/` back into the project.

Edit the configuration cell to run a smaller slice. `ogbn-arxiv` is not in the list: the dense pairwise step does not fit that graph. S30 is GCN, GAT, and APPNP, which are already listed.


In [ ]:
# --- configuration ---------------------------------------------------------
REPO_URL = "https://github.com/TAR2003/RUNG_ML_Project.git"
BRANCH = "master"

DATASETS = [
    "cora", "citeseer", "pubmed",
    "chameleon", "cornell", "wisconsin",
    "squirrel", "actor", "texas",
]

MODELS = [
    "RUNG",
    "RUNG_percentile_gamma",
    "RUNG_learnable_distance",
    "RUNG_parametric_gamma",
    "RUNG_learnable_gamma",
    "RUNG_learnable_combined",
    "RUNG_confidence_lambda",
    "RUNG_homophily_adaptive",
    "RUNG_combined",
    "RUNG_combined_model",
    "GCN", "GAT", "APPNP", "MLP",
    "S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S09",
    "S10", "S11", "S12", "S13", "S14", "S15", "S16", "S17", "S18", "S19",
    "S20", "S21", "S22", "S23", "S24", "S25", "S26", "S27", "S28", "S29",
    "S31", "S32", "S33", "S34",
]

MAX_EPOCH = 300
RUN_ATTACK = True
RUN_COSINE = True
SKIP_FINISHED = True


In [ ]:
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"

ON_KAGGLE = Path("/kaggle/working").is_dir()
INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else None
WORK = Path("/kaggle/working/RUNG_ML_Project") if ON_KAGGLE else Path.cwd() / "kaggle_work" / "RUNG_ML_Project"


def _find_attached_project():
    if INPUT is None:
        return None
    hits = []
    for run_all in INPUT.rglob("run_all.py"):
        root = run_all.parent
        if (root / "model" / "rung_strategy.py").is_file():
            hits.append(root)
    if not hits:
        return None
    hits.sort(key=lambda p: (p / "log").is_dir(), reverse=True)
    return hits[0]


def _materialize():
    if (WORK / "run_all.py").is_file() and (WORK / "model" / "rung_strategy.py").is_file():
        print("Using checkout already in", WORK)
        return
    attached = _find_attached_project()
    if attached is not None:
        print("Copying attached project from", attached)
        shutil.copytree(
            attached,
            WORK,
            ignore=shutil.ignore_patterns(".git", "__pycache__", ".ipynb_checkpoints"),
            dirs_exist_ok=True,
        )
        return
    WORK.parent.mkdir(parents=True, exist_ok=True)
    print("Cloning", REPO_URL, BRANCH)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(WORK)]
    )


def _restore_previous_results():
    if INPUT is None:
        return
    for archive in INPUT.rglob("rung_results.zip"):
        print("Restoring previous logs, figures, and checkpoints from", archive)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORK)


_materialize()
_restore_previous_results()

if not (WORK / "model" / "rung_strategy.py").is_file():
    raise SystemExit(
        "model/rung_strategy.py is not in the checkout.\n"
        "Push the local RUNG project to https://github.com/TAR2003/RUNG_ML_Project "
        "and re-run, or attach a Kaggle dataset that contains the project."
    )

os.chdir(WORK)
os.environ["PYTHONPATH"] = str(WORK)
print("Project:", WORK)

skip = {"torch", "torchvision", "torchaudio", "seml"}
pkgs = []
for line in (WORK / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    name = re.split(r"[=<>]", line, maxsplit=1)[0].strip().lower()
    if name not in skip:
        pkgs.append(line)

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "typeguard==2.13.3", *pkgs, "torch_geometric"]
)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit(
        "No GPU is visible. In the Kaggle notebook settings set Accelerator to GPU, then Run All again."
    )


In [ ]:
unit = subprocess.run([sys.executable, str(WORK / "tests" / "test_rung_strategy.py")], cwd=WORK)
print("unit tests", "passed" if unit.returncode == 0 else "FAILED")
if unit.returncode != 0:
    print("Unit checks failed. The algorithm loop still runs so one failure does not hide the rest.")


In [ ]:
def log_stem(model: str) -> str:
    """Filename stem written by get_log_identifier for run_all.py defaults."""
    if re.fullmatch(r"S\d+", model):
        return f"RUNG_strategy_{model}"
    stems = {
        "RUNG": "RUNG_normMCP_gamma6.0",
        "MLP": "MLP_normMCP_gamma6.0",
        "GCN": "GCN_normMCP_gamma6.0",
        "GAT": "GAT_normMCP_gamma6.0",
        "APPNP": "APPNP_normMCP_gamma6.0",
        "RUNG_percentile_gamma": "RUNG_percentile_gamma_q0.75",
        "RUNG_learnable_distance": "RUNG_learnable_distance_distcosine_q0.75",
        "RUNG_parametric_gamma": "RUNG_parametric_gamma_decay0.85_reg0.0",
        "RUNG_learnable_gamma": "RUNG_learnable_gamma_inituniform_reg0.0",
        "RUNG_learnable_combined": "RUNG_learnable_combined_modeper_layer",
        "RUNG_combined": "RUNG_combined_q0.75",
        "RUNG_combined_model": "RUNG_combined_model_q0.75_decay0.85_a0.5",
        "RUNG_confidence_lambda": "RUNG_confidence_lambda_prot_a1.0",
        "RUNG_homophily_adaptive": "RUNG_homophily_adaptive_normMCP_gamma6.0",
    }
    return stems[model]


def _log_has(dataset: str, kind: str, model: str, needle: str) -> bool:
    path = WORK / "log" / dataset / kind / f"{log_stem(model)}.log"
    if not path.is_file() or path.stat().st_size < 40:
        return False
    text = path.read_text(encoding="utf-8", errors="ignore")
    return needle in text


def phase_flags(dataset: str, model: str):
    clean_ok = _log_has(dataset, "clean", model, "clean acc:")
    attack_ok = (not RUN_ATTACK) or _log_has(dataset, "attack", model, "Attacked:")
    cosine_ok = (not RUN_COSINE) or _log_has(dataset, "cosine_attack", model, "Budget:")
    return clean_ok, attack_ok, cosine_ok


rows = []
for model in MODELS:
    for dataset in DATASETS:
        clean_ok, attack_ok, cosine_ok = phase_flags(dataset, model)
        if SKIP_FINISHED and clean_ok and attack_ok and cosine_ok:
            print(f"skip  {dataset:12} {model}  (already finished)")
            rows.append({"dataset": dataset, "model": model, "status": "skipped"})
            continue

        cmd = [
            sys.executable, str(WORK / "run_all.py"),
            "--datasets", dataset,
            "--models", model,
            "--max_epoch", str(MAX_EPOCH),
        ]
        if clean_ok:
            cmd.append("--skip_clean")
        if attack_ok:
            cmd.append("--skip_attack")
        if RUN_COSINE and not cosine_ok:
            cmd.append("--cosine-attack")

        print("\n$", " ".join(cmd), flush=True)
        proc = subprocess.run(cmd, cwd=WORK)
        clean_ok, attack_ok, cosine_ok = phase_flags(dataset, model)
        status = "ok" if proc.returncode == 0 and clean_ok and attack_ok and cosine_ok else "failed"
        print(f"{status}  {dataset} {model}", flush=True)
        rows.append({"dataset": dataset, "model": model, "status": status})

n_ok = sum(r["status"] in ("ok", "skipped") for r in rows)
print(f"\n{n_ok}/{len(rows)} dataset-algorithm pairs finished or already finished")
for r in rows:
    if r["status"] == "failed":
        print("FAILED", r["dataset"], r["model"])


In [ ]:
chart = subprocess.run(
    [sys.executable, str(WORK / "plot_logs.py"), "--log_dir", "log", "--out_dir", "figures"],
    cwd=WORK,
)
print("charts", "written" if chart.returncode == 0 else "FAILED")

archive = Path("/kaggle/working/rung_results.zip") if ON_KAGGLE else WORK.parent / "rung_results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in ("log", "figures", "exp/models"):
        root = WORK / folder
        if not root.exists():
            continue
        for file in root.rglob("*"):
            if file.is_file():
                zf.write(file, file.relative_to(WORK).as_posix())
print("Saved", archive)

from IPython.display import Image, Markdown, display

pngs = sorted((WORK / "figures").rglob("*.png"))
if not pngs:
    print("No charts yet. Training logs appear in", WORK / "log")
for png in pngs:
    display(Markdown(f"**{png.relative_to(WORK).as_posix()}**"))
    display(Image(filename=str(png)))
